# TrustVision — Exp B: XAI, Calibration & Adversarial Robustness
**Module:** IA Avancée · ENSA Fès  
**Rôle:** MEMBRE B (Thèmes 9 & 11)  
**Objectif:** Boîte à outils d'audit de confiance appliquée aux checkpoints du pipeline :
1. **Thème 9 (IA explicable et fiable):** Grad-CAM, Integrated Gradients avec complétude, calibration ECE avec bootstrap IC 95%, Temperature Scaling optimisé sur calibration (180 images) et fiabilité des diagrammes.
2. **Thème 11 (ML adversarial):** Attaques FGSM avec balayage $\varepsilon \in \{0, 0.01, 0.03, 0.05, 0.10\}$, défense par entraînement adversarial, compromis exactitude/robustesse, et stabilité intra-pipeline des explications.


In [ ]:
# 1. Automatic Package Installation & Imports (Google Colab & Local Execution)
import sys
import subprocess
import os
import shutil
import zipfile
import json
import random
import hashlib
from pathlib import Path
from typing import Dict, Any, List, Tuple, Optional, Union

def install_if_missing(package_name: str, import_name: Optional[str] = None):
    name = import_name or package_name
    try:
        __import__(name)
    except ImportError:
        print(f"[Setup] Installing {package_name}...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", package_name])

# Ensure required libraries
install_if_missing("torch")
install_if_missing("torchvision")
install_if_missing("numpy")
install_if_missing("scipy")
install_if_missing("scikit-learn", "sklearn")
install_if_missing("matplotlib")
install_if_missing("Pillow", "PIL")
install_if_missing("requests")

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt

print(f"[Setup] PyTorch Version: {torch.__version__}")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"[Setup] Compute Device: {device} (Estimated run time: ~2 min on CPU, ~30s on GPU)")


In [ ]:
# 2. Strict Determinism and Reproducibility Setup
SEED = 42

def set_seed(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

set_seed(SEED)
print(f"[Setup] Deterministic random seed locked to {SEED}.")


In [ ]:
# 3. Load NEU-CLS Dataset via Fixed Splits (70/10/20)
from src.data import get_dataloaders, NEU_CLASSES

train_loader, cal_loader, test_loader, test_files = get_dataloaders(
    root_dir="data/NEU-CLS",
    batch_size=32,
    seed=SEED,
    img_size=96,
    auto_download=True,
)

print(f"[Data] Classes: {NEU_CLASSES}")
print(f"[Data] Train samples: {len(train_loader.dataset)} (1 260 images)")
print(f"[Data] Cal samples:   {len(cal_loader.dataset)} (180 images)")
print(f"[Data] Test samples:  {len(test_loader.dataset)} (360 images)")
assert len(train_loader.dataset) == 1260 and len(cal_loader.dataset) == 180 and len(test_loader.dataset) == 360


In [ ]:
# 4. Load Baseline CNN (cnn_base_seed42.pt) and Verify Contracts
from src.eval import load_model_checkpoint
from src.model import TrustVisionCNN

base_ckpt_path = "checkpoints/cnn_base_seed42.pt"
base_model = load_model_checkpoint(base_ckpt_path, device=device)
print(f"[Model] Successfully loaded {base_ckpt_path}")

# Verify shared contracts: get_features and get_last_conv_layer
dummy_batch = torch.rand(2, 3, 96, 96).to(device)
with torch.no_grad():
    feats = base_model.get_features(dummy_batch)
    last_conv = base_model.get_last_conv_layer()

print(f"[Model Contract] get_features shape: {feats.shape} (expected 256-d for SimCLR)")
print(f"[Model Contract] get_last_conv_layer: {last_conv} (used by Grad-CAM)")


In [ ]:
# 5. XAI Audit: Grad-CAM vs Integrated Gradients (Completeness & Agreement)
from src.xai_cal_adv import GradCAM, integrated_gradients, compare_xai_methods

xai_comparison = compare_xai_methods(base_model, test_loader, device=device, max_samples=60)
print(f"[XAI] Mean Cosine Similarity (Grad-CAM vs IG): {xai_comparison['mean_cosine_similarity']:.4f} ± {xai_comparison['std_cosine_similarity']:.4f}")
print(f"[XAI] Mean IG Relative Completeness Error:     {xai_comparison['mean_ig_completeness_rel_error']:.4f}")

# Visual Comparison on Test Samples
images_sample, labels_sample = next(iter(test_loader))
images_sample = images_sample[:4].to(device)
labels_sample = labels_sample[:4].to(device)

gradcam_explainer = GradCAM(base_model)
cams = gradcam_explainer(images_sample)
gradcam_explainer.remove_hooks()
igs, comp_stats = integrated_gradients(base_model, images_sample, steps=30)

fig, axes = plt.subplots(4, 3, figsize=(10, 12))
for i in range(4):
    img_np = images_sample[i].permute(1, 2, 0).cpu().numpy()
    true_cls = NEU_CLASSES[labels_sample[i].item()]
    
    # Original
    axes[i, 0].imshow(img_np)
    axes[i, 0].set_title(f"Image ({true_cls})")
    axes[i, 0].axis("off")
    
    # Grad-CAM
    axes[i, 1].imshow(img_np)
    axes[i, 1].imshow(cams[i], cmap="jet", alpha=0.55)
    axes[i, 1].set_title("Grad-CAM Overlay")
    axes[i, 1].axis("off")
    
    # Integrated Gradients
    axes[i, 2].imshow(igs[i], cmap="hot")
    axes[i, 2].set_title("Integrated Gradients")
    axes[i, 2].axis("off")

plt.tight_layout()
plt.show()


In [ ]:
# 6. XAI Cross-Check: Grad-CAM Alignment with CLIP ViT Attention Map (S0)
from src.eval import compute_xai_similarity

clip_attn_file = Path("results/clip_attn_test.npy")
if clip_attn_file.exists():
    vit_maps = np.load(str(clip_attn_file))
    if vit_maps.ndim == 4:
        vit_maps = vit_maps.mean(axis=1)  # (360, 7, 7)
    
    # Compute Grad-CAM on all test samples
    gradcam_explainer = GradCAM(base_model)
    all_cams = []
    for imgs, _ in test_loader:
        all_cams.append(gradcam_explainer(imgs.to(device)))
    gradcam_explainer.remove_hooks()
    all_cams_cat = np.concatenate(all_cams, axis=0)
    
    sim_scores = [
        compute_xai_similarity(all_cams_cat[i], vit_maps[i], method="cosine")
        for i in range(len(test_files))
    ]
    mean_vit_sim = np.mean(sim_scores)
    print(f"[XAI] Mean Cosine Agreement between CNN Grad-CAM and CLIP ViT Attention: {mean_vit_sim:.4f}")
    
    # Plot side-by-side comparison for first sample
    fig, axes = plt.subplots(1, 3, figsize=(11, 3.5))
    axes[0].imshow(all_cams_cat[0], cmap="jet")
    axes[0].set_title("CNN Grad-CAM (96x96)")
    axes[0].axis("off")
    
    axes[1].imshow(vit_maps[0], cmap="viridis")
    axes[1].set_title("CLIP ViT Patch Attention (7x7)")
    axes[1].axis("off")
    
    from src.eval import _resize_explanation_map
    resized_cam = _resize_explanation_map(all_cams_cat[0], (7, 7))
    axes[2].imshow(resized_cam, cmap="jet")
    axes[2].set_title("Grad-CAM Projected to ViT (7x7)")
    axes[2].axis("off")
    plt.tight_layout()
    plt.show()
else:
    print("[XAI] results/clip_attn_test.npy not found, skipping cross-comparison.")


In [ ]:
# 7. Calibration Analysis: Reliability Diagrams & Temperature Scaling
from src.xai_cal_adv import fit_and_evaluate_temperature_scaling

ts_results = fit_and_evaluate_temperature_scaling(base_model, cal_loader, test_loader, device=device)

print(f"[Calibration] Optimal Temperature T (fitted on cal): {ts_results['temperature']:.4f}")
print(f"[Calibration] Argmax strictly invariant:            {ts_results['argmax_invariant']}")
print(f"[Calibration] Uncalibrated ECE: {ts_results['uncalibrated']['ece']:.4f} (95% CI: {ts_results['uncalibrated']['ece_ci']})")
print(f"[Calibration] Calibrated ECE:   {ts_results['calibrated']['ece']:.4f} (95% CI: {ts_results['calibrated']['ece_ci']})")
print(f"[Calibration] Test Accuracy:    {ts_results['uncalibrated']['accuracy'] * 100:.2f}% (strictly identical)")

# Plot Reliability Diagrams
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
bin_edges = np.linspace(0, 1, 11)
centers = (bin_edges[:-1] + bin_edges[1:]) / 2

for ax, key, title in zip(axes, ["uncalibrated", "calibrated"], ["Before Temperature Scaling", "After Temperature Scaling"]):
    stats = ts_results[key]["bin_stats"]
    accs = stats["accuracies"]
    confs = stats["confidences"]
    ece_val = ts_results[key]["ece"]
    
    ax.bar(centers, accs, width=0.08, alpha=0.7, color="royalblue", edgecolor="black", label="Outputs (Acc)")
    ax.plot([0, 1], [0, 1], "--", color="red", label="Perfect Calibration")
    ax.set_xlim([0, 1])
    ax.set_ylim([0, 1])
    ax.set_xlabel("Confidence")
    ax.set_ylabel("Accuracy")
    ax.set_title(f"{title}\nECE = {ece_val:.4f}")
    ax.legend(loc="upper left")
    ax.grid(True, linestyle=":", alpha=0.5)

plt.tight_layout()
plt.show()


In [ ]:
# 8. FGSM Attack: Evaluating Robustness Vulnerability of the Standard CNN
from src.xai_cal_adv import evaluate_fgsm_sweep

eps_list = [0.0, 0.01, 0.03, 0.05, 0.10]
clean_sweep = evaluate_fgsm_sweep(base_model, test_loader, epsilons=eps_list, device=device)

print("[FGSM] Standard CNN Accuracy vs Epsilon budget:")
for eps in eps_list:
    acc = clean_sweep[str(eps)]['accuracy']
    print(f"  eps = {eps:.2f} : {acc * 100:.2f}%")


In [ ]:
# 9. Adversarial Training Defense & Robustness Comparison
from src.xai_cal_adv import compare_clean_vs_adversarial

adv_ckpt_path = "checkpoints/cnn_adv_seed42.pt"
comp_results = compare_clean_vs_adversarial(
    clean_checkpoint=base_ckpt_path,
    adv_checkpoint=adv_ckpt_path,
    test_loader=test_loader,
    device=device,
    epsilons=eps_list,
    compute_explanation_stability=True,
    n_xai_samples=60,
)

print(f"[Robustness] Clean Baseline Rob Acc (eps=0.03): {comp_results['clean_rob_acc_003'] * 100:.2f}%")
print(f"[Robustness] Adversarial CNN Rob Acc (eps=0.03): {comp_results['adv_rob_acc_003'] * 100:.2f}%")
print(f"[Robustness] Net Robustness Gain at eps=0.03:    +{comp_results['robustness_gain_003'] * 100:.2f}%")

# Plot FGSM Robust Accuracy Curves
plt.figure(figsize=(8, 5))
clean_accs = [comp_results['clean_model_sweep'][str(e)]['accuracy'] * 100 for e in eps_list]
adv_accs = [comp_results['adv_model_sweep'][str(e)]['accuracy'] * 100 for e in eps_list]

plt.plot(eps_list, clean_accs, "o-", color="firebrick", linewidth=2.2, label="Clean Baseline CNN (S2)")
plt.plot(eps_list, adv_accs, "s-", color="forestgreen", linewidth=2.2, label="Adversarial Trained CNN (S3)")
plt.axvline(x=0.03, color="gray", linestyle="--", alpha=0.7, label="Contract Budget eps=0.03")

plt.xlabel("FGSM Perturbation Budget (epsilon) on pixels [0, 1]", fontsize=11)
plt.ylabel("Test Set Accuracy (%)", fontsize=11)
plt.title("Robustness Curve: Clean vs. Adversarially Defended CNN", fontsize=12)
plt.legend(fontsize=10)
plt.grid(True, linestyle=":", alpha=0.6)
plt.ylim([0, 105])
plt.show()


In [ ]:
# 10. Defense Effect on Explanations: Intra-Pipeline Stability
xai_stab = comp_results['explanation_stability']
print(f"[XAI Stability] Grad-CAM Cosine Stability (Clean vs Adv): {xai_stab['gradcam_stability_cosine']:.4f}")
print(f"[XAI Stability] IG Cosine Stability (Clean vs Adv):       {xai_stab['ig_stability_cosine']:.4f}")

# Visual Comparison of Saliency Maps Before vs After Defense
adv_model = load_model_checkpoint(adv_ckpt_path, device=device)
gradcam_clean = GradCAM(base_model)
gradcam_adv = GradCAM(adv_model)

imgs, lbls = next(iter(test_loader))
imgs_sub = imgs[:3].to(device)
cam_clean_vis = gradcam_clean(imgs_sub)
cam_adv_vis = gradcam_adv(imgs_sub)
gradcam_clean.remove_hooks()
gradcam_adv.remove_hooks()

fig, axes = plt.subplots(3, 3, figsize=(10, 9))
for i in range(3):
    img_np = imgs_sub[i].permute(1, 2, 0).cpu().numpy()
    axes[i, 0].imshow(img_np)
    axes[i, 0].set_title(f"Original ({NEU_CLASSES[lbls[i].item()]})")
    axes[i, 0].axis("off")
    
    axes[i, 1].imshow(img_np)
    axes[i, 1].imshow(cam_clean_vis[i], cmap="jet", alpha=0.55)
    axes[i, 1].set_title("Grad-CAM (Clean CNN)")
    axes[i, 1].axis("off")
    
    axes[i, 2].imshow(img_np)
    axes[i, 2].imshow(cam_adv_vis[i], cmap="jet", alpha=0.55)
    axes[i, 2].set_title("Grad-CAM (Adv CNN)")
    axes[i, 2].axis("off")

plt.tight_layout()
plt.show()


In [ ]:
# 11. Cross-Effects Table (S0 to S4 Pipeline Overview)
from src.run_experiments import build_cross_effects_table

matrix = build_cross_effects_table(seed=SEED, device=device)
print(f"\n{'Stage':<28} | {'Acc':<7} | {'Macro-F1':<8} | {'ECE':<7} | {'Rob@0.03':<8} | {'XAI Sim':<7} | {'Intra Stab':<10} | {'Status'}")
print("-" * 105)
for row in matrix["rows"]:
    acc_s = f"{row['acc']*100:.1f}%" if row['acc'] is not None else "-"
    f1_s = f"{row['macro_f1']:.3f}" if row['macro_f1'] is not None else "-"
    ece_s = f"{row['ece']:.3f}" if row['ece'] is not None else "-"
    rob_s = f"{row['rob_acc_003']*100:.1f}%" if row['rob_acc_003'] is not None else "-"
    xai_s = f"{row['xai_sim']:.3f}" if row['xai_sim'] is not None else "-"
    intra_s = f"{row['intra_xai_stab']:.3f}" if row['intra_xai_stab'] is not None else "-"
    print(f"{row['stage']:<28} | {acc_s:<7} | {f1_s:<8} | {ece_s:<7} | {rob_s:<8} | {xai_s:<7} | {intra_s:<10} | {row['status']}")


In [ ]:
# 12. Fast Inference Function for Gradio Tab 3 (< 0.5s)
import time
from src.eval import TemperatureScaler, fgsm_attack

class GradioInferenceAudit:
    def __init__(self, model_checkpoint: str = "checkpoints/cnn_base_seed42.pt", device: Optional[torch.device] = None):
        if device is None:
            device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        self.device = device
        self.model = load_model_checkpoint(model_checkpoint, device=self.device)
        self.gradcam = GradCAM(self.model)
        self.scaler = TemperatureScaler(initial_temp=1.0)
        self.classes = NEU_CLASSES

    def audit_sample(self, image_tensor: torch.Tensor, temperature: float = 1.0, epsilon: float = 0.0) -> Dict[str, Any]:
        """
        Runs fast inference on a test image with temperature scaling and FGSM perturbation.
        Runs in < 0.5 seconds on CPU.
        """
        t0 = time.time()
        if image_tensor.ndim == 3:
            image_tensor = image_tensor.unsqueeze(0)
        image_tensor = image_tensor.to(self.device)

        # 1. FGSM Attack
        if epsilon > 0.0:
            with torch.no_grad():
                init_pred = self.model(image_tensor).argmax(dim=1)
            perturbed = fgsm_attack(self.model, image_tensor, init_pred, eps=epsilon)
        else:
            perturbed = image_tensor.clone()

        # 2. Forward pass & Temperature Scaling
        with torch.no_grad():
            raw_logits = self.model(perturbed)
            calibrated_logits = raw_logits / max(temperature, 1e-3)
            probs = F.softmax(calibrated_logits, dim=1).squeeze(0)
            pred_idx = int(probs.argmax().item())
            confidence = float(probs[pred_idx].item())

        # 3. Grad-CAM Explanation
        cam_map = self.gradcam(perturbed, target_classes=[pred_idx])[0]

        elapsed = time.time() - t0
        return {
            "predicted_class": self.classes[pred_idx],
            "confidence": round(confidence, 4),
            "probabilities": {c: round(float(probs[i].item()), 4) for i, c in enumerate(self.classes)},
            "grad_cam_map": cam_map,
            "latency_seconds": round(elapsed, 4),
        }

# Smoke test
auditor = GradioInferenceAudit(base_ckpt_path, device=device)
test_img, _ = test_loader.dataset[0]
res = auditor.audit_sample(test_img, temperature=1.2, epsilon=0.03)
print(f"[Gradio Tab 3 Ready] Pred: {res['predicted_class']} ({res['confidence']*100:.1f}%) in {res['latency_seconds']:.3f} s")
